### Notebook to calculate coverage and length using classical methods like cvx and PLS

In [2]:
import numpy as np
import torch
import torch.nn as nn
from tqdm import tqdm
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Dataset, TensorDataset, random_split
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import random
from uq_utils import *
import pickle
import seaborn as sns
from models import *
from plot_util import *

# Check if MPS is available
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print('using device:', device) 

import sys
import os

using device: mps


In [4]:
test_methods = ['CVX', 'PLS']
             
    
n_qubits_list = [2]
n_shots_list = [50, 100]
n_reps = 2
alpha = 0.1
# save_path = "/Users/aditi/Documents/Fall_25/Quantum/UQ/main/results/conformal_NN"
# os.makedirs(save_path, exist_ok=True)
save_path = "./results"
os.makedirs(save_path, exist_ok=True)

records = []
save = True

for nQubits in n_qubits_list:
    for nShots in n_shots_list:
        for i in range(n_reps):

            sigma = np.random.randn(2**nQubits, 1) + 1j*np.random.randn(2**nQubits, 1)
            sigma_vec = sigma/np.linalg.norm(sigma)
            sigma = sigma @ sigma.T.conj()
            sigma /= np.trace(sigma)

            train_states = [HS_state(N= 2**nQubits) for _ in range(10000)]
            X = [counts(rho, nQubits, nShots) for rho in train_states]
            Y = [np.real(fidelity(rho, sigma)) for rho in train_states]

            X = np.asarray(X, dtype=np.float32)
            Y = np.asarray(Y, dtype=np.float32)

            # divide the dataset into test and train based on the test_ratio parameter
            x_train, x_test, y_train, y_test = train_test_split(X,
                                                                Y.ravel(),
                                                                test_size=0.2,
                                                                random_state=i)


            # compute input dimensions
            n_train = x_train.shape[0]
            n_test = x_test.shape[0]
            in_shape = x_train.shape[1]
            

            # divide the data into proper training set and calibration set
            idx = np.random.permutation(n_train)
            n_half = int(np.floor(n_train/2))
            idx_train, idx_cal = idx[:n_half], idx[n_half:2*n_half]
            n_cal = idx_cal.shape[0]

            for method in test_methods:

                if method == "PLS":
                    solver = PLS(nQubits=nQubits)
                elif method == "CVX":
                    solver = CVX(nQubits=nQubits)

                print(f"\n# qubits:{nQubits}, iteration:{i}, method:{method}")
                fidelities_cal, _ = solver.batch_estimate(x_train[idx_cal], sigma)
                # print(np.mean(np.abs(y_train[idx_cal]- fidelities_cal)))
                # print(y_train[idx_cal], fidelities_cal)
                
                scores = np.abs(fidelities_cal.real-y_train[idx_cal])
                # print(scores)
                q_hat = np.quantile(scores, np.ceil((n_cal+1)*(1-alpha))/n_cal, method='inverted_cdf')   
                
                fidelities_test, _ = solver.batch_estimate(x_test, sigma)
                coverage, length = compute_coverage(y_test, fidelities_test.real - q_hat, fidelities_test.real + q_hat)
                print(coverage, length)
                records.append({
                    "nQubits": nQubits,
                    "nShots": nShots,
                    "iter": i,
                    "method": method,
                    'coverage': coverage*100,
                    'avg_length': length
                    })

df_results = pd.DataFrame(records)
if save:
    df_results.to_csv(os.path.join(save_path, "conformal_classic_all_methods.csv"), index=False)



# qubits:2, iteration:0, method:CVX
0.9065 0.17060054306814293

# qubits:2, iteration:0, method:PLS
0.9055 0.16917665727620806

# qubits:2, iteration:1, method:CVX


/opt/miniconda3/envs/qutip-env/lib/python3.11/site-packages/scs/__init__.py:83: UserWarning: Converting A to a CSC (compressed sparse column) matrix; may take a while.
  warn(
/opt/miniconda3/envs/qutip-env/lib/python3.11/site-packages/scs/__init__.py:83: UserWarning: Converting A to a CSC (compressed sparse column) matrix; may take a while.
  warn(
/opt/miniconda3/envs/qutip-env/lib/python3.11/site-packages/scs/__init__.py:83: UserWarning: Converting A to a CSC (compressed sparse column) matrix; may take a while.
  warn(
/opt/miniconda3/envs/qutip-env/lib/python3.11/site-packages/scs/__init__.py:83: UserWarning: Converting A to a CSC (compressed sparse column) matrix; may take a while.
  warn(
/opt/miniconda3/envs/qutip-env/lib/python3.11/site-packages/scs/__init__.py:83: UserWarning: Converting A to a CSC (compressed sparse column) matrix; may take a while.
  warn(


0.9055 0.17288106462986816

# qubits:2, iteration:1, method:PLS
0.9035 0.1719851043366371

# qubits:2, iteration:0, method:CVX


/opt/miniconda3/envs/qutip-env/lib/python3.11/site-packages/scs/__init__.py:83: UserWarning: Converting A to a CSC (compressed sparse column) matrix; may take a while.
  warn(
/opt/miniconda3/envs/qutip-env/lib/python3.11/site-packages/scs/__init__.py:83: UserWarning: Converting A to a CSC (compressed sparse column) matrix; may take a while.
  warn(
/opt/miniconda3/envs/qutip-env/lib/python3.11/site-packages/scs/__init__.py:83: UserWarning: Converting A to a CSC (compressed sparse column) matrix; may take a while.
  warn(


0.8885 0.12600254752890144

# qubits:2, iteration:0, method:PLS
0.8885 0.12575309997581163

# qubits:2, iteration:1, method:CVX


/opt/miniconda3/envs/qutip-env/lib/python3.11/site-packages/scs/__init__.py:83: UserWarning: Converting A to a CSC (compressed sparse column) matrix; may take a while.
  warn(
/opt/miniconda3/envs/qutip-env/lib/python3.11/site-packages/scs/__init__.py:83: UserWarning: Converting A to a CSC (compressed sparse column) matrix; may take a while.
  warn(


0.901 0.12803328383892465

# qubits:2, iteration:1, method:PLS
0.9015 0.12798973295501032


In [9]:
summary = df_results.groupby(["nShots", "method"]).agg(
        mean_len=("avg_length", "mean")
        )

print(summary)
    

               mean_len
nShots method          
50     CVX     0.172398
       PLS     0.171843
100    CVX     0.128196
       PLS     0.127796
500    CVX     0.060431
       PLS     0.060418
